# 03. Tomato Field Video Surveillance Pipeline

This notebook demonstrates the multi-task video surveillance pipeline, integrating calibrated ResNet18 disease diagnosis with YOLOv8n-Seg pest damage segmentation across temporal field video frames.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.surveillance.video_processor import VideoProcessor
from src.surveillance.frame_analyzer import SurveillanceFrameAnalyzer
from src.surveillance.annotator import VideoAnnotator
from src.surveillance.timeline_plotter import plot_surveillance_timeline

video_path = PROJECT_ROOT / "data" / "video" / "sample_field_surveillance.mp4"
print(f"Target Video: {video_path}")

In [ ]:
# 1. Validate video and inspect metadata
processor = VideoProcessor(max_duration_seconds=60.0, max_file_size_mb=100.0)
meta = processor.validate_and_inspect(video_path)
print(f"Resolution: {meta.width}x{meta.height} | FPS: {meta.fps} | Frames: {meta.frame_count} | Duration: {meta.duration_seconds:.2f}s")

In [ ]:
# 2. Run multi-task frame analysis with calibrated ResNet18 and YOLOv8n-Seg
analyzer = SurveillanceFrameAnalyzer(temperature=1.3021, use_yolov8=True)
annotator = VideoAnnotator()

results = []
for f_idx, ts, frame_bgr in processor.sample_frames(video_path, stride=5, max_frames=6):
    res = analyzer.analyze_frame(frame_bgr, frame_idx=f_idx, timestamp=ts)
    results.append(res)
    print(f"Frame {f_idx} (t={ts:.2f}s): {res.disease_label} ({res.disease_confidence:.2%}), {res.pest_count} pest mines")

In [ ]:
# 3. Plot temporal progression timeline
timeline_fig_path = Path("/tmp/nb_surveillance_timeline.png")
plot_surveillance_timeline(results, save_path=timeline_fig_path)
print(f"Timeline exported to: {timeline_fig_path}")